In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
# Première URL
url = 'https://www.diabete.qc.ca/le-diabete-en-questions/'
response = requests.get(url)
if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    my_dict = {}
    sections = soup.find_all('section', class_='accordion')
    for section in sections:
        answers = []
        question = section.find('span', class_='accordion__header__title').text.strip()
        results = section.find('div', class_='accordion__sub-rows').find_all('p')
        for result in results:
            answers.append(result.text.strip())
        my_dict[question] = ' '.join(answers)
    
    # Conversion en DataFrame
    df = pd.DataFrame(my_dict.items(), columns=['questions', 'answers'])
else:
    print(f"Erreur de chargement : {response.status_code}")


In [3]:
# Deuxième URL
base_url = 'https://www.diabete.qc.ca/le-diabete/informations-sur-le-diabete/'
URNs = [
    'quest-ce-que-le-diabete', 'facteurs-de-risque', 'symptomes',
    'depistage-et-diagnostic', 'diabete-de-type-1', 'diabete-de-type-2',
    'diabete-de-grossesse', 'prediabete', 'autres-types-de-diabete'
]

my_second_dict = {}
pattern = r'[\xa0\n]'

for uri in URNs:
    try:
        full_url = f'{base_url}{uri}'
        response = requests.get(full_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        question = soup.find('h1')
        question_text = question.text.strip() if question else f"Page sans titre: {uri}"

        answer = soup.find('div', class_='entry-content')
        answer_text = re.sub(pattern, ' ', answer.text.strip()) if answer else "Contenu non trouvé"

        my_second_dict[question_text] = answer_text

    except Exception as e:
        print(f"Erreur sur {uri} : {e}")

In [4]:
# Conversion en DataFrame et fusion des données
df_second = pd.DataFrame(my_second_dict.items(), columns=['questions', 'answers'])
data = pd.concat([df, df_second], ignore_index=True)

In [5]:
# Sauvegarde des données dans un fichier CSV
data.to_csv('diabetes_data.csv', index=False)

In [6]:
# Affichage du DataFrame final
data.head()


,questions,answers
0,Qu'est-ce que le diabète ?,Le diabète est une maladie chronique qui ne se...
1,Quelles sont les complications du diabète ?,Les complications liées au diabète ont une ori...
2,Quels sont les symptômes ?,Les symptômes du diabète peuvent varier d’une ...
3,Quels sont les principaux types de diabète ?,Le diabète de type 1 Le diabète de type 1 se m...
4,Quels sont les facteurs de risques du diabète ...,Les causes du diabète de type 2 sont nombreuse...
